# Enrichment and pathway analysis of the 30 Boruta genes

Reads `pd-lcm-rf-core`, `pd-lcm-rf-boruta-panel` and `pd-lcm-rf-external-multi`; nothing there is refitted.

1. **Over-representation** - up-regulated (18), down-regulated (12) and all 30 genes, analysed separately as the primary
   analysis (up- and down-regulated genes usually reflect different biology) and together as a secondary one.
   Background: the 5,622 genes actually measured. Terms of 10-500 genes, BH over **every** tested term, and each result
   calibrated against random gene sets of the same size and against random sets matched on differential-expression strength.
2. **Dopamine-neuron subtypes** - Kamath et al. 2022 (Nat Neurosci) found that the SOX6_AGTR1 dopamine neurons degenerate
   in PD while CALB1 neurons are relatively spared. Do the panel's down genes mark the vulnerable SOX6 lineage and its up
   genes the resilient CALB1 lineage?
3. **Pathway neighbours** - for each pathway that contains a panel gene, do the pathway's *other* genes move the same way in
   the discovery donors? Label shuffles within study; whole-search FWER and empirical FDR.
4. **Per-gene table** - function, known PD association (Open Targets), subtype lineage, replication in the 8 bulk cohorts.

In [ ]:
import os, io, re, json, time, glob, zipfile, urllib.request, urllib.parse, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import hypergeom, mannwhitneyu, spearmanr, rankdata
from scipy import sparse
warnings.filterwarnings("ignore")
ON_KAGGLE = Path("/kaggle/input").exists()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("SMOKE_OUT", "smoke_enr"))
OUT.mkdir(parents=True, exist_ok=True)
B_RAND = 2000 if ON_KAGGLE else 50          # random / matched gene sets per list
B_PERM = 5000 if ON_KAGGLE else 60          # label shuffles
rng = np.random.default_rng(42)
t0 = time.time()
def log(m): print(f"[{time.time() - t0:5.0f}s] {m}", flush=True)
def find_any(pattern, key):
    root = "/kaggle/input" if ON_KAGGLE else os.environ["LOCAL_" + key.upper().replace("-", "_")]
    hits = sorted((h for h in glob.glob(f"{root}/**/{pattern}", recursive=True) if (key in h or not ON_KAGGLE)), key=len)
    if not hits:
        raise FileNotFoundError(f"{pattern} ({key})")
    return hits[0]
def get(url, data=None, headers=None, timeout=300, tries=5):
    for k in range(tries):
        try:
            req = urllib.request.Request(url, data=data, headers=headers or {"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as fh:
                return fh.read()
        except Exception as exc:
            print("  retry", k + 1, type(exc).__name__); time.sleep(8)
    raise RuntimeError(url)
pd.set_option("display.width", 230); pd.set_option("display.max_colwidth", 60)

## 0. The discovery data, the panel and the background

In [ ]:
cz = np.load(find_any("core_data.npz", "rf-core"), allow_pickle=True)
X, y, DS = cz["X"].astype(float), cz["y"].astype(int), cz["ds"].astype(str)
GENES = [str(g) for g in cz["genes"]]
SYM = pd.read_csv(find_any("15_gene_symbol_map.csv", "rf-core")).set_index("gene")["symbol"].to_dict()
DE = pd.read_csv(find_any("03_de_results_full.csv", "boruta-panel")).set_index("gene").reindex(GENES)
ANN = pd.read_csv(find_any("13_gene_annotation_master.csv", "boruta-panel")).set_index("gene")
PANEL = pd.read_csv(find_any("04_boruta_selected_genes.csv", "boruta-panel")).gene.tolist()
EXT = pd.read_csv(find_any("external_multi_gene_meta.csv", "external-multi")).set_index("gene")
sym = [SYM.get(g) if isinstance(SYM.get(g), str) else None for g in GENES]
BG = pd.DataFrame({"gene": GENES, "symbol": [s.upper() if s else None for s in sym], "g": DE.hedges_g_meta.to_numpy(),
                   "p": DE.pvalue.to_numpy()})
BG = BG[BG.symbol.notna() & ~BG.symbol.duplicated()].reset_index(drop=True)
IDX = {s: i for i, s in enumerate(BG.symbol)}
GI = {g: i for i, g in enumerate(BG.gene)}
N = len(BG)
UP = [g for g in PANEL if DE.loc[g, "hedges_g_meta"] > 0]
DOWN = [g for g in PANEL if DE.loc[g, "hedges_g_meta"] < 0]
LISTS = {"up": UP, "down": DOWN, "all": PANEL}
S = lambda gs: [SYM.get(g, g) for g in gs]
log(f"{len(y)} donors, background {N:,} genes with symbols; panel {len(PANEL)}: {len(UP)} up, {len(DOWN)} down")
print("up:  ", ", ".join(S(UP))); print("down:", ", ".join(S(DOWN)))
PANEL_I = np.array([GI[g] for g in PANEL])
# matching for the DE-matched null: 10 bins of |g| x sign, panel genes excluded from the pools
BG["bin"] = pd.qcut(BG.g.abs().rank(method="first"), 10, labels=False).astype(int) * 2 + (BG.g > 0).astype(int)
POOLS = {b: np.setdiff1d(np.flatnonzero(BG.bin.to_numpy() == b), PANEL_I) for b in BG.bin.unique()}
def matched_draw(idx):
    out = []
    for b, k in pd.Series(BG.bin.to_numpy()[idx]).value_counts().items():
        out.extend(rng.choice(POOLS[b], k, replace=False))
    return np.array(out)

## 1. Gene-set libraries

In [ ]:
LIBS = ["GO_Biological_Process_2023", "GO_Cellular_Component_2023", "GO_Molecular_Function_2023", "Reactome_2022",
        "KEGG_2021_Human", "WikiPathway_2023_Human"]
SHORT = {"GO_Biological_Process_2023": "GO:BP", "GO_Cellular_Component_2023": "GO:CC", "GO_Molecular_Function_2023": "GO:MF",
         "Reactome_2022": "Reactome", "KEGG_2021_Human": "KEGG", "WikiPathway_2023_Human": "WikiPathways"}
def enrichr_lib(name):
    txt = get(f"https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName={name}").decode()
    out = {}
    for line in txt.splitlines():
        parts = line.split("\t")
        if len(parts) > 2:
            out[parts[0].strip()] = {g.split(",")[0].strip().upper() for g in parts[2:] if g.strip()}
    return out
if ON_KAGGLE:
    RAW = {lib: enrichr_lib(lib) for lib in LIBS}
else:                                            # smoke run: random sets plus a few holding panel genes
    syms = BG.symbol.to_numpy(); RAW = {}
    for lib in LIBS:
        RAW[lib] = {f"{lib[:6]} term {i}": set(rng.choice(syms, rng.integers(12, 300), replace=False)) for i in range(150)}
        for i, g in enumerate(S(PANEL)[:8]):
            RAW[lib][f"{lib[:6]} anchored {i}"] = set(rng.choice(syms, 40, replace=False)) | {g.upper()}
TERMS, seen = [], set()
for lib in LIBS:
    for term, genes in RAW[lib].items():
        idx = tuple(sorted(IDX[g] for g in genes if g in IDX))
        if len(idx) < 10 or idx in seen:                              # identical gene sets across libraries kept once
            continue
        seen.add(idx)
        TERMS.append({"library": SHORT[lib], "term": re.sub(r"\s*\((GO:\d+)\)$", r" (\1)", term), "idx": np.array(idx),
                      "size": len(idx)})
TERMS = pd.DataFrame(TERMS)
def member_matrix(T):
    rows = np.repeat(np.arange(len(T)), T["size"].to_numpy())
    return sparse.csr_matrix((np.ones(len(rows), dtype=np.int32), (rows, np.concatenate(T.idx.to_numpy()))), shape=(len(T), N))
log(f"{len(TERMS):,} distinct gene sets with >= 10 background genes: " + ", ".join(f"{k} {v}" for k, v in TERMS.library.value_counts().items()))

## 2. Over-representation: up, down, all

Hypergeometric test against the 5,622 measured genes, terms of 10-500 background genes, BH over **all** terms (not only
those the list touches - selecting the family on the outcome is what makes small queries return false pathways).
Calibration: the same procedure on 2,000 random gene sets of the same size, and on 2,000 sets matched gene by gene on
differential-expression strength and direction.

In [ ]:
ORA_T = TERMS[TERMS["size"] <= 500].reset_index(drop=True)
MB = member_matrix(ORA_T)
K = ORA_T["size"].to_numpy()
def bh(p):
    p = np.asarray(p, float); o = np.argsort(p); q = p[o] * len(p) / np.arange(1, len(p) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]; out = np.empty_like(q); out[o] = np.minimum(q, 1); return out
def ora_table(n):
    """P value lookup by (hits, term) for a list of n genes: hypergeometric upper tail."""
    return np.vstack([hypergeom.sf(h - 1, N, K, n) for h in range(n + 1)])
def hits_of(sets):
    R = np.zeros((N, len(sets)), dtype=np.int32)
    for j, s in enumerate(sets):
        R[s, j] = 1
    return np.asarray(MB @ R)
ORA, CAL = [], {}
for name, genes in LISTS.items():
    idx = np.array([GI[g] for g in genes]); n = len(idx); P = ora_table(n)
    h = hits_of([idx])[:, 0]; p = P[h, np.arange(len(K))]; q = bh(p)
    rnd = [rng.choice(N, n, replace=False) for _ in range(B_RAND)]
    mat = [matched_draw(idx) for _ in range(B_RAND)]
    hr, hm = hits_of(rnd), hits_of(mat)                                # terms x draws
    pr, pm = P[hr, np.arange(len(K))[:, None]], P[hm, np.arange(len(K))[:, None]]
    minr = np.where(hr >= 2, pr, 1).min(0); minm = np.where(hm >= 2, pm, 1).min(0)
    fp_any = np.mean([(bh(pr[:, j]) < 0.05)[hr[:, j] >= 2].any() for j in range(B_RAND)])
    CAL[name] = {"n": n, "terms": len(K), "random_sets_with_any_BH_hit": float(fp_any),
                 "p_fwer05_random": float(np.quantile(minr, 0.05)), "p_fwer05_matched": float(np.quantile(minm, 0.05))}
    keep = np.flatnonzero(h >= 2)
    for t in keep:
        members = set(ORA_T.idx[t]); gl = [BG.symbol[i] for i in idx if i in members]
        ORA.append({"list": name, "library": ORA_T.library[t], "term": ORA_T.term[t], "size": int(K[t]), "hits": int(h[t]),
                    "genes": ", ".join(sorted(gl)), "p": p[t], "q_BH": q[t],
                    "fwer_random": (np.sum(minr <= p[t]) + 1) / (B_RAND + 1),
                    "p_vs_matched": (np.sum(pm[t] <= p[t]) + 1) / (B_RAND + 1),
                    "fwer_matched": (np.sum(minm <= p[t]) + 1) / (B_RAND + 1)})
    log(f"{name}: {n} genes, {len(K):,} terms, {len(keep)} with >= 2 genes; best p = {p[keep].min() if len(keep) else 1:.2e}; "
        f"BH < 0.05: {int((q[keep] < 0.05).sum())}; random sets with any BH hit: {100 * fp_any:.1f}%")
ORA = pd.DataFrame(ORA).sort_values(["list", "p"]).reset_index(drop=True)
ORA.to_csv(OUT / "enrich_ora.csv", index=False)
for name in LISTS:
    print(f"\n==== {name}"); print(ORA[ORA.list == name].head(8)[["library", "term", "size", "hits", "genes", "p", "q_BH", "fwer_random",
                                                                    "p_vs_matched"]].to_string(index=False))

## 3. Per-gene PD effect in the discovery donors, observed and under label shuffles

The same statistic for every gene - a Hedges' g within each study, averaged across the four studies with weights
n1*n0/(n1+n0) - computed on the real labels and on labels shuffled within each study. Sections 4 and 5 use it.

In [ ]:
XB = X[:, [GENES.index(g) for g in BG.gene]]
STUDIES = sorted(set(DS))
def effect(Ys):
    """Ys: (b x donors) 0/1 labels -> (b x genes) pooled within-study Hedges' g."""
    num = np.zeros((Ys.shape[0], XB.shape[1]), np.float32); den = 0.0
    for d in STUDIES:
        m = DS == d; Xk = XB[m].astype(np.float32); Yk = Ys[:, m].astype(np.float32)
        n1 = Yk.sum(1, keepdims=True); n0 = m.sum() - n1
        S1, Q1 = Yk @ Xk, Yk @ (Xk ** 2); S, Q = Xk.sum(0), (Xk ** 2).sum(0)
        m1, m0 = S1 / n1, (S - S1) / n0
        v1 = (Q1 - n1 * m1 ** 2) / (n1 - 1); v0 = (Q - Q1 - n0 * m0 ** 2) / (n0 - 1)
        sp = np.sqrt(np.clip(((n1 - 1) * v1 + (n0 - 1) * v0) / (n1 + n0 - 2), 1e-6, None))
        J = 1 - 3 / (4 * (n1 + n0) - 9); w = float((n1 * n0 / (n1 + n0))[0, 0])
        num += w * J * (m1 - m0) / sp; den += w
    return num / den
G_OBS = effect(y[None, :])[0]
def shuffled(b):
    Ys = np.tile(y, (b, 1))
    for d in STUDIES:
        m = np.flatnonzero(DS == d)
        Ys[:, m] = np.array([rng.permutation(y[m]) for _ in range(b)])
    return Ys
G_PERM = np.vstack([effect(shuffled(min(500, B_PERM - s))) for s in range(0, B_PERM, 500)])
log(f"per-gene effect: observed vs the DE table's meta g, Spearman {spearmanr(G_OBS, BG.g)[0]:.3f}; {G_PERM.shape[0]:,} label shuffles")

## 4. Dopamine-neuron subtypes (Kamath et al. 2022)

Supplementary Table 8 of Kamath et al. gives, for each of ten human dopamine-neuron subtypes, the genes that mark it against
the other dopamine neurons (MAST z). **Lineage score** of a gene = its mean z across the six CALB1 subtypes minus its mean z
across the four SOX6 subtypes: positive marks the resilient CALB1 lineage, negative the vulnerable SOX6 lineage (SOX6_AGTR1 is
the population lost in PD).

In [ ]:
import zipfile, re, xml.etree.ElementTree as ET
import pandas as pd
def read_strict_xlsx(path):
    """Every sheet of an .xlsx workbook as a DataFrame - handles 'strict' OOXML, which openpyxl cannot open."""
    zx = zipfile.ZipFile(path)
    wb = ET.fromstring(zx.read("xl/workbook.xml"))
    M = wb.tag.split("}")[0].strip("{")                                  # main namespace, strict or transitional
    rels = ET.fromstring(zx.read("xl/_rels/workbook.xml.rels"))
    target = {r.get("Id"): r.get("Target") for r in rels}
    RID = [k for k in wb.find(f"{{{M}}}sheets")[0].attrib if k.endswith("}id")][0]
    ss = []
    if "xl/sharedStrings.xml" in zx.namelist():
        for si in ET.fromstring(zx.read("xl/sharedStrings.xml")).findall(f"{{{M}}}si"):
            ss.append("".join(t.text or "" for t in si.iter(f"{{{M}}}t")))
    col = lambda ref: sum((ord(ch) - 64) * 26 ** i for i, ch in enumerate(reversed(re.match(r"[A-Z]+", ref).group())))
    out = {}
    for sh in wb.find(f"{{{M}}}sheets"):
        f = "xl/" + target[sh.get(RID)].lstrip("/").replace("xl/", "")
        rows = []
        for r in ET.fromstring(zx.read(f)).iter(f"{{{M}}}row"):
            vals = {}
            for c in r.findall(f"{{{M}}}c"):
                v = c.findtext(f"{{{M}}}v")
                if c.get("t") == "s" and v is not None:
                    v = ss[int(v)]
                elif c.get("t") == "inlineStr":
                    v = "".join(t.text or "" for t in c.iter(f"{{{M}}}t"))
                vals[col(c.get("r")) - 1] = v
            rows.append([vals.get(i) for i in range(max(vals) + 1)] if vals else [])
        out[sh.get("name")] = pd.DataFrame(rows)
    return out


In [ ]:
if ON_KAGGLE:
    j = json.loads(get("https://www.ebi.ac.uk/europepmc/webservices/rest/search?format=json&query=" + urllib.parse.quote('DOI:"10.1038/s41593-022-01061-1"')))
    pmcid = j["resultList"]["result"][0]["pmcid"]
    zf = zipfile.ZipFile(io.BytesIO(get(f"https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/supplementaryFiles", timeout=600)))
    name = [n for n in zf.namelist() if n.endswith("MOESM3_ESM.xlsx")][0]
    (OUT / "kamath_2022_supplementary.xlsx").write_bytes(zf.read(name))
    T8 = read_strict_xlsx(OUT / "kamath_2022_supplementary.xlsx")["Supplementary_Table_8"]
    T8.columns = [str(c) if c is not None else f"c{i}" for i, c in enumerate(T8.iloc[0])]; T8 = T8.iloc[1:]
    T8 = T8.rename(columns={"primerid": "symbol"})[["DA_subtype", "symbol", "coef", "z", "fdr"]]
else:                                                  # smoke run: invented markers
    subs = ["SOX6_AGTR1", "SOX6_PART1", "SOX6_DDT", "SOX6_GFRA2", "CALB1_CALCR", "CALB1_CRYM_CCDC68", "CALB1_GEM", "CALB1_PPP1R17",
            "CALB1_RBP4", "CALB1_TRHR"]
    T8 = pd.DataFrame([{"DA_subtype": s, "symbol": g, "coef": rng.normal(), "z": rng.normal(0, 4), "fdr": rng.uniform(0, 0.05)}
                       for s in subs for g in rng.choice(BG.symbol, 1500, replace=False)])
for c in ("coef", "z", "fdr"):
    T8[c] = pd.to_numeric(T8[c], errors="coerce")
T8["symbol"] = T8.symbol.astype(str).str.upper()
print(f"Kamath Table 8: {len(T8):,} rows, {T8.symbol.nunique():,} genes, subtypes {sorted(T8.DA_subtype.unique())}; "
      f"FDR range {T8.fdr.min():.1e}-{T8.fdr.max():.2f}; coef<0 rows {int((T8.coef < 0).sum()):,}")
ZT = T8.pivot_table(index="symbol", columns="DA_subtype", values="z", aggfunc="first").reindex(BG.symbol).fillna(0.0)
SOX_SUB = [c for c in ZT.columns if c.startswith("SOX6")]; CALB_SUB = [c for c in ZT.columns if c.startswith("CALB1")]
BG["lineage"] = (ZT[CALB_SUB].mean(1) - ZT[SOX_SUB].mean(1)).to_numpy()
BG["agtr1"] = ZT["SOX6_AGTR1"].to_numpy()
cover = float((ZT.abs().sum(1) > 0).mean())
lin = BG.lineage.to_numpy()
SUB = {"coverage": cover}
up_i, dn_i = np.array([GI[g] for g in UP]), np.array([GI[g] for g in DOWN])
SUB["up_vs_down"] = mannwhitneyu(lin[up_i], lin[dn_i], alternative="greater").pvalue
SUB["up_vs_background"] = mannwhitneyu(lin[up_i], np.delete(lin, PANEL_I), alternative="greater").pvalue
SUB["down_vs_background"] = mannwhitneyu(lin[dn_i], np.delete(lin, PANEL_I), alternative="less").pvalue
# DE-matched null: is the separation larger than for genes equally strongly changed in PD?
obs = lin[up_i].mean() - lin[dn_i].mean()
null = np.array([lin[matched_draw(up_i)].mean() - lin[matched_draw(dn_i)].mean() for _ in range(B_RAND)])
SUB["separation"] = float(obs); SUB["p_vs_DE_matched"] = float((np.sum(null >= obs) + 1) / (B_RAND + 1))
# transcriptome-wide: do genes of the CALB1 lineage rise and genes of the SOX6 lineage fall in PD neurons?
rho = spearmanr(G_OBS, lin)[0]
rho_null = np.array([spearmanr(G_PERM[b], lin)[0] for b in range(min(G_PERM.shape[0], 2000))])
SUB["rho_transcriptome"] = float(rho); SUB["rho_p_label_shuffle"] = float((np.sum(np.abs(rho_null) >= abs(rho)) + 1) / (len(rho_null) + 1))
# each subtype's top-200 markers, against the up and the down list
SUBTAB = []
for st in ZT.columns:
    mk = T8[(T8.DA_subtype == st) & (T8.coef > 0) & (T8.fdr < 0.05) & T8.symbol.isin(IDX)].nlargest(200, "z").symbol
    mk_i = set(IDX[s] for s in mk)
    for name, gi in (("up", up_i), ("down", dn_i)):
        h = int(sum(i in mk_i for i in gi))
        SUBTAB.append({"subtype": st, "lineage": st.split("_")[0], "list": name, "markers": len(mk_i), "hits": h,
                       "genes": ", ".join(sorted(BG.symbol[i] for i in gi if i in mk_i)),
                       "p": hypergeom.sf(h - 1, N, len(mk_i), len(gi))})
SUBTAB = pd.DataFrame(SUBTAB); SUBTAB["q_BH"] = bh(SUBTAB.p)
SUBTAB.to_csv(OUT / "enrich_subtypes.csv", index=False)
PG = BG.loc[PANEL_I, ["gene", "symbol", "g", "lineage", "agtr1"]].assign(direction=lambda d: np.where(d.g > 0, "up", "down"))
PG.to_csv(OUT / "enrich_panel_lineage.csv", index=False)
BG[["gene", "symbol", "g", "lineage", "agtr1"]].to_csv(OUT / "enrich_background_lineage.csv", index=False)
SUB["background_p05"], SUB["background_p95"] = float(np.percentile(lin, 5)), float(np.percentile(lin, 95))
print(json.dumps(SUB, indent=1, default=float))
print(PG.sort_values("lineage").round(2).to_string(index=False))
print(SUBTAB.sort_values("p").head(12).round(4).to_string(index=False))

## 5. Pathway neighbours

Every term of 10-150 background genes that holds at least one panel gene. The anchor's direction is the sign of its panel
genes' summed effect; the statistic is the mean effect of the term's **other** genes (all panel genes removed), signed by the
anchor's direction. Null: the same statistic on 5,000 within-study label shuffles; whole-search FWER from the minimum p
across terms in each shuffle, and empirical FDR.

In [ ]:
PANEL_SET = set(PANEL_I)
CT = []
for t in TERMS[TERMS["size"] <= 150].itertuples():
    anchors = [i for i in t.idx if i in PANEL_SET]
    nb = [i for i in t.idx if i not in PANEL_SET]
    if anchors and len(nb) >= 5:
        d = np.sign(sum(G_OBS[i] for i in anchors))
        if d != 0:
            CT.append({"library": t.library, "term": t.term, "size": t.size, "anchors": ", ".join(BG.symbol[i] for i in anchors),
                       "direction": "up" if d > 0 else "down", "d": d, "nb": np.array(nb)})
CT = pd.DataFrame(CT)
NBM = sparse.csr_matrix((np.ones(sum(len(v) for v in CT.nb)), (np.repeat(np.arange(len(CT)), [len(v) for v in CT.nb]), np.concatenate(CT.nb))),
                        shape=(len(CT), N))
nn = np.array([len(v) for v in CT.nb]); dd = CT.d.to_numpy()
obs = dd * (NBM @ G_OBS) / nn
null = (np.asarray(NBM @ G_PERM.T.astype(float)).T / nn) * dd                 # shuffles x terms
p_obs = (np.sum(null >= obs, axis=0) + 1) / (null.shape[0] + 1)
p_null = (null.shape[0] - rankdata(null, axis=0, method="min") + 1) / null.shape[0]          # each shuffle's own p, per term
minp = p_null.min(1)
CT["neighbours"] = nn; CT["shift"] = obs; CT["p"] = p_obs
CT["fwer"] = [(np.sum(minp <= p) + 1) / (len(minp) + 1) for p in p_obs]
order = np.argsort(p_obs); fdr = np.empty(len(p_obs))
for rank, t in enumerate(order, 1):
    fdr[t] = min(1.0, np.mean((p_null <= p_obs[t]).sum(1)) / rank)
CT["fdr"] = np.minimum.accumulate(fdr[order][::-1])[::-1][np.argsort(order)]
pct = rankdata(G_OBS) / N
CT["neighbour_median_percentile"] = [float(np.median(pct[v] if d > 0 else 1 - pct[v])) for v, d in zip(CT.nb, CT.d)]
CT["top_neighbours"] = [", ".join(BG.symbol[i] for i in v[np.argsort(-d * G_OBS[v])][:6]) for v, d in zip(CT.nb, CT.d)]
CT = CT.drop(columns=["d", "nb"]).sort_values("p").reset_index(drop=True)
CT.to_csv(OUT / "enrich_pathway_neighbours.csv", index=False)
log(f"pathway neighbours: {len(CT)} terms hold a panel gene; best p {CT.p.min():.1e}; FWER < 0.05: {int((CT.fwer < 0.05).sum())}; "
    f"empirical FDR < 0.05: {int((CT.fdr < 0.05).sum())}, < 0.10: {int((CT.fdr < 0.10).sum())}")
print(CT.head(15)[["library", "term", "size", "anchors", "direction", "neighbours", "shift", "p", "fwer", "fdr", "neighbour_median_percentile",
                   "top_neighbours"]].round(4).to_string(index=False))

## 6. Per-gene table

In [ ]:
ids = list(PANEL)
if ON_KAGGLE:
    mg = json.loads(get("https://mygene.info/v3/query", data=urllib.parse.urlencode(
        {"q": ",".join(ids), "scopes": "ensembl.gene", "fields": "symbol,name,summary", "species": "human"}).encode()))
    MG = {x["query"]: x for x in mg if not x.get("notfound")}
    q = """query($ids:[String!]!){ disease(efoId:"MONDO_0005180"){ associatedTargets(Bs:$ids, page:{index:0,size:100}){ rows{
          target{ id } score datatypeScores{ id score } } } } }"""
    ot = json.loads(get("https://api.platform.opentargets.org/api/v4/graphql", data=json.dumps({"query": q, "variables": {"ids": ids}}).encode(),
                        headers={"Content-Type": "application/json"}))
    OT = {r["target"]["id"]: {"ot_score": r["score"], **{f"ot_{d['id']}": d["score"] for d in r["datatypeScores"]}}
          for r in ot["data"]["disease"]["associatedTargets"]["rows"]}
else:
    MG, OT = {}, {}
first = lambda s: (re.split(r"(?<=[.])\s", s)[0] if isinstance(s, str) else "")
best_nb = {}
for r in CT.itertuples():
    for a in r.anchors.split(", "):
        best_nb.setdefault(a, r)
rows = []
for g in PANEL:
    a = ANN.loc[g] if g in ANN.index else pd.Series(dtype=float)
    e = EXT.loc[g] if g in EXT.index else pd.Series(dtype=float)
    s_ = BG.symbol[GI[g]]; b = BG.loc[GI[g]]; nbr = best_nb.get(s_)
    rows.append({"gene": g, "symbol": SYM.get(g, g), "name": MG.get(g, {}).get("name", ""),
                 "direction": "up" if DE.loc[g, "hedges_g_meta"] > 0 else "down", "hedges_g_discovery": DE.loc[g, "hedges_g_meta"],
                 "p_discovery": DE.loc[g, "pvalue"], "deg": bool(DE.loc[g, "is_deg"]),
                 "boruta_fold_frequency": a.get("boruta_fold_frequency"), "shap_rank": a.get("shap_rank"), "single_gene_auc": a.get("single_gene_auc"),
                 "g_external_neuron_adjusted": e.get("g_external_neuron_adj"),
                 "replicates_in_bulk": bool(np.sign(e.get("g_external_neuron_adj", np.nan)) == np.sign(DE.loc[g, "hedges_g_meta"])),
                 "dopamine_lineage_score": b.lineage,
                 "top_subtype_marker": (ZT.loc[s_].idxmax() if ZT.loc[s_].max() > 0 else ""),
                 "pd_association_open_targets": OT.get(g, {}).get("ot_score", 0.0),
                 "pd_genetic_association": OT.get(g, {}).get("ot_genetic_association", 0.0),
                 "pd_literature": OT.get(g, {}).get("ot_literature", 0.0),
                 "best_pathway_neighbours": f"{nbr.term} ({nbr.library}; p={nbr.p:.1e}, FDR={nbr.fdr:.2f})" if nbr is not None else "",
                 "summary": first(MG.get(g, {}).get("summary", ""))})
GT = pd.DataFrame(rows).sort_values(["direction", "hedges_g_discovery"], ascending=[False, False])
GT.to_csv(OUT / "enrich_gene_table.csv", index=False)
try:
    GT.to_excel(OUT / "enrich_gene_table.xlsx", index=False)
except Exception as exc:
    print("xlsx not written:", exc)
print(GT.drop(columns=["gene", "summary"]).round(3).to_string(index=False))
json.dump({"lists": {k: S(v) for k, v in LISTS.items()}, "ora_calibration": CAL, "subtypes": SUB,
           "neighbours": {"terms": int(len(CT)), "fwer_lt_0.05": int((CT.fwer < 0.05).sum()), "fdr_lt_0.05": int((CT.fdr < 0.05).sum()),
                          "fdr_lt_0.10": int((CT.fdr < 0.10).sum())}},
          open(OUT / "enrich_summary.json", "w"), indent=1, default=float)
log("done")

## 7. Figure (print size, 183 x 128 mm)

In [ ]:
import matplotlib
try:
    get_ipython(); IN_NB = True
except NameError:
    IN_NB = False; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Rectangle, Polygon, FancyBboxPatch
from matplotlib.path import Path as MPath
from matplotlib.patches import PathPatch

INK, MUTED, RULE = "#1B1D20", "#5E656D", "#C9CED4"
PD_C, CT_C = "#7A2533", "#6F829A"
CORE_C, PANEL_C, NEURO_C = "#23324A", "#8E1B2E", "#9AA1A9"
SETS = {"both": ("#8E1B2E", "#5E0F1C", "Boruta & DEG"), "boruta": ("#2E5A87", "#1B3A5C", "Boruta only")}
EFFECT = LinearSegmentedColormap.from_list("effect", ["#1E3350", "#4C6583", "#9AAABB", "#EEECE7", "#C3A09E", "#8C4B52", "#551C28"])
FONT_STACK = ["Helvetica Neue", "Helvetica", "Arial", "Liberation Sans", "Nimbus Sans", "FreeSans", "DejaVu Sans"]
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": FONT_STACK, "font.size": 6.3,
                     "axes.linewidth": 0.5, "axes.edgecolor": "#30343A", "axes.labelcolor": INK, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "xtick.labelsize": 5.9, "ytick.labelsize": 5.9,
                     "xtick.major.width": 0.5, "ytick.major.width": 0.5, "xtick.major.size": 2.1, "ytick.major.size": 2.1,
                     "xtick.major.pad": 1.7, "ytick.major.pad": 1.7, "xtick.color": "#30343A", "ytick.color": "#30343A",
                     "pdf.fonttype": 42, "ps.fonttype": 42, "figure.dpi": 150, "savefig.facecolor": "white",
                     "figure.facecolor": "white", "mathtext.fontset": "custom", "mathtext.rm": "sans", "mathtext.it": "sans:italic"})

class Canvas:
    """A print-size figure drawn on a millimetre grid."""
    def __init__(self, w, h):
        self.W, self.H = w, h
        self.fig = plt.figure(figsize=(w / 25.4, h / 25.4))
        self.M = self.fig.add_axes([0, 0, 1, 1]); self.M.set_xlim(0, w); self.M.set_ylim(0, h); self.M.axis("off")
    def ax(self, x, y, w, h):
        return self.fig.add_axes([x / self.W, y / self.H, w / self.W, h / self.H])
    def letter(self, x, y, L, title):
        self.M.text(x, y, L, ha="left", va="baseline", fontsize=9, fontweight="bold", color=INK)
        self.M.text(x + 4.2, y, title, ha="left", va="baseline", fontsize=7.2, color=INK)
    def rule(self, x0, x1, y, lw=0.4, color=RULE):
        self.M.plot([x0, x1], [y, y], color=color, lw=lw, solid_capstyle="butt")
    def text(self, x, y, s, **kw):
        kw.setdefault("va", "center"); kw.setdefault("fontsize", 6.0)
        return self.M.text(x, y, s, **kw)
    def save(self, stem):
        for ext in ("pdf", "png"):
            self.fig.savefig(OUT / f"{stem}.{ext}", dpi=600 if ext == "png" else None)
        print("saved", stem)
        plt.show() if IN_NB else plt.close(self.fig)
f2 = lambda v: f"{v:.2f}"
def pfmt(p): return f"= {p:.3f}" if p >= 0.001 else "< 0.001"   # "P = 0.003", "P < 0.0001"


In [ ]:
import os, re, json
from pathlib import Path
import numpy as np, pandas as pd
from matplotlib.lines import Line2D
FIG_IN = Path(os.environ.get("FIG_IN", str(OUT)))
PLN = pd.read_csv(FIG_IN / "enrich_panel_lineage.csv")
BGL = pd.read_csv(FIG_IN / "enrich_background_lineage.csv")
SBT = pd.read_csv(FIG_IN / "enrich_subtypes.csv")
ORT = pd.read_csv(FIG_IN / "enrich_ora.csv")
GTB = pd.read_csv(FIG_IN / "enrich_gene_table.csv").set_index("gene")
SMY = json.load(open(os.environ.get("FIG_SUMMARY", str(FIG_IN / "enrich_summary.json"))))
SU = SMY["subtypes"]
NICE = GTB.symbol.to_dict()
UPPER2NICE = {v.upper(): v for v in GTB.symbol}
UPC, DNC, ALLC = PANEL_C, CORE_C, "#4A4F56"

C = Canvas(183.0, 128.0); M = C.M

# ======================= a: the lineage axis =======================
C.letter(2.0, 124.0, "a", "Panel genes on the dopamine-neuron lineage axis")
X0, W = 24.0, 150.0
LT = 2.0                                                   # linear part of the symmetric-log scale
tr = lambda v: np.sign(v) * np.log10(1 + np.abs(v) / LT)  # symmetric log, used for placement and bins
lo, hi = tr(-22.0), tr(19.0)
axT = C.ax(X0, 88.0, W, 25.0); axT.set_xlim(lo, hi); axT.set_ylim(0, 2.3); axT.axis("off")
axH = C.ax(X0, 79.0, W, 8.0); axH.set_xlim(lo, hi)
for ax in (axT, axH):
    ax.axvspan(lo, tr(SU["background_p05"]), color="#EEF1F4", lw=0, zorder=0)
    ax.axvspan(tr(SU["background_p95"]), hi, color="#EEF1F4", lw=0, zorder=0)
    ax.axvline(0, color="#C9CED4", lw=0.5, zorder=0)
bins = np.linspace(lo, hi, 90)
hh, _ = np.histogram(tr(BGL.lineage.to_numpy()), bins=bins)
axH.fill_between(np.repeat(bins, 2)[1:-1], 0, np.repeat(np.sqrt(hh), 2), color="#B9BEC4", lw=0, zorder=1)
axH.set_ylim(0, np.sqrt(hh).max() * 1.05); axH.set_yticks([]); axH.spines["left"].set_visible(False)
TK = [-16, -8, -4, -2, 0, 2, 4, 8, 16]
axH.set_xticks([tr(v) for v in TK]); axH.set_xticklabels([f"{v:+d}" if v else "0" for v in TK]); axH.tick_params(axis="x", labelsize=5.8)
axH.patch.set_alpha(0)
C.text(X0 - 1.5, 83.0, f"All {len(BGL):,} genes", ha="right", fontsize=5.7, color=MUTED)
C.text(X0 + W / 2, 71.0, "Lineage score: mean marker z in CALB1 subtypes minus SOX6 subtypes (Kamath et al. 2022)", ha="center", fontsize=6.0)
C.text(X0, 71.0, "$\\leftarrow$ SOX6 lineage (vulnerable)", ha="left", fontsize=5.8, color=DNC)
C.text(X0 + W, 71.0, "CALB1 lineage (resilient) $\\rightarrow$", ha="right", fontsize=5.8, color=UPC)
TRACK = {"down": 1.55, "up": 0.55}
for d, col, lab in (("down", DNC, "Down in PD"), ("up", UPC, "Up in PD")):
    sub = PLN[PLN.direction == d]
    yv = TRACK[d]
    axT.plot([lo, hi], [yv, yv], color="#DADDE1", lw=0.5, zorder=1)
    axT.scatter(tr(sub.lineage.to_numpy()), np.full(len(sub), yv), s=16, color=col, edgecolor="white", linewidth=0.4, zorder=4)
    C.text(X0 - 1.5, 88.0 + 25.0 * yv / 2.3, f"{lab} ({len(sub)})", ha="right", fontsize=6.0, color=col, fontweight="bold")
    # labels for the genes beyond the central band, staggered so they never touch
    lab_df = sub[sub.lineage.abs() > 2.5].assign(x=lambda d_: tr(d_.lineage)).sort_values("x")
    mm_per = W / (hi - lo); last = {}
    for r in lab_df.itertuples():
        level = 0
        while level in last and (r.x - last[level]) * mm_per < 9.0:
            level += 1
        last[level] = r.x
        ty = yv + (0.22 + 0.2 * level) * (1 if d == "down" else -1)
        axT.plot([r.x, r.x], [yv + 0.06 * np.sign(ty - yv), ty - 0.04 * np.sign(ty - yv)], color="#9AA1A9", lw=0.4, zorder=2)
        axT.text(r.x, ty, NICE.get(r.gene, r.symbol), ha="center", va="bottom" if d == "down" else "top", fontsize=5.8,
                 fontstyle="italic", color=col)
pf = lambda p: f"P = {p:.4f}" if p >= 0.0001 else "P < 0.0001"
C.text(181.0, 124.0, f"Down vs up genes: {pf(SU['up_vs_down'])} (Mann-Whitney)", ha="right", fontsize=5.8)
C.text(181.0, 120.8, f"vs genes equally changed in PD: {pf(SU['p_vs_DE_matched'])}", ha="right", fontsize=5.8)
C.text(181.0, 117.6, "shaded: outer 5% of all genes", ha="right", fontsize=5.5, color=MUTED)
C.rule(2.0, 181.0, 66.5, lw=0.35, color="#8C9299")

# ======================= b: subtype markers =======================
C.letter(2.0, 61.5, "b", "Overlap with subtype markers")
ORDER = ["SOX6_AGTR1", "SOX6_PART1", "SOX6_DDT", "SOX6_GFRA2", "CALB1_CALCR", "CALB1_CRYM_CCDC68", "CALB1_GEM", "CALB1_PPP1R17",
         "CALB1_RBP4", "CALB1_TRHR"]
CX = {"down": 44.0, "up": 56.0}
RY0, RS = 49.0, 4.2
C.text(CX["down"], 55.0, "Down\n(12)", ha="center", fontsize=5.8, color=DNC, fontweight="bold", linespacing=1.0)
C.text(CX["up"], 55.0, "Up\n(18)", ha="center", fontsize=5.8, color=UPC, fontweight="bold", linespacing=1.0)
C.rule(4.0, 70.0, 52.4, lw=0.5, color=INK)
sbt = SBT.set_index(["subtype", "list"])
for i, st in enumerate(ORDER):
    yy = RY0 - i * RS - (1.6 if i >= 4 else 0)
    C.text(28.5, yy, st.replace("_", " "), ha="right", fontsize=5.7)
    for d in ("down", "up"):
        r = sbt.loc[(st, d)]; col = DNC if d == "down" else UPC
        if r.hits:
            sig = r.q_BH < 0.05
            M.scatter([CX[d]], [yy], s=6 + 9 * r.hits, color=col if sig else "white", edgecolor=col, linewidth=0.8, zorder=3)
            C.text(CX[d] + 3.2, yy, str(int(r.hits)), ha="left", fontsize=5.3, color=col if sig else MUTED)
        else:
            M.plot([CX[d] - 0.6, CX[d] + 0.6], [yy, yy], color="#C9CED4", lw=0.6)
    if st == "SOX6_AGTR1":
        C.text(63.0, yy, "lost in PD", ha="left", fontsize=5.3, color=MUTED, fontstyle="italic")
for grp, i0, i1 in (("SOX6", 0, 3), ("CALB1", 4, 9)):
    y0 = RY0 - i0 * RS - (1.6 if i0 >= 4 else 0) + 1.6; y1 = RY0 - i1 * RS - (1.6 if i1 >= 4 else 0) - 1.6
    M.plot([4.8, 4.8], [y1, y0], color=DNC if grp == "SOX6" else UPC, lw=1.4, solid_capstyle="butt")
    C.text(3.2, (y0 + y1) / 2, grp, ha="center", rotation=90, fontsize=5.6, color=DNC if grp == "SOX6" else UPC, fontweight="bold")
LY = RY0 - 9 * RS - 1.6 - 6.0
for k, (s_, lab) in enumerate(((1, "1"), (2, "2"), (4, "4 genes"))):
    M.scatter([8.0 + k * 7.5], [LY], s=6 + 9 * s_, color="white", edgecolor="#6B7178", linewidth=0.7)
    C.text(9.8 + k * 7.5, LY, lab, ha="left", fontsize=5.3, color=MUTED)
M.scatter([38.0], [LY], s=24, color="#6B7178", linewidth=0); C.text(40.0, LY, "filled: FDR < 0.05 (200 top markers)", ha="left", fontsize=5.3, color=MUTED)

# ======================= c: over-representation, against a calibrated threshold =======================
C.letter(78.0, 61.5, "c", "Pathway over-representation")
def nice_term(t):
    t = re.sub(r"\s*\((GO:\d+)\)$", "", t); t = re.sub(r"\s+R-HSA-\d+$", "", t); t = re.sub(r"\s+WP\d+$", "", t)
    t = t[0] + t[1:].lower()
    for a in ("dna", "rna", "gpcr", "hsf1", "cns"):
        t = re.sub(rf"\b{a}\b", a.upper(), t)
    return t if len(t) <= 50 else t[:48].rstrip() + "..."
AXX0, AXW = 126.0, 30.0
GX = AXX0 + AXW + 2.0
axC = C.ax(AXX0, 7.5, AXW, 44.5)
ROWS, yy, GROUPS = [], 0.0, []
for lst, col, lab in (("up", UPC, "Up in PD (18 genes)"), ("down", DNC, "Down in PD (12 genes)"), ("all", ALLC, "All 30 genes")):
    top = ORT[ORT.list == lst].nsmallest(4, "p")
    y_start = yy
    for r in top.itertuples():
        ROWS.append((yy, r, col)); yy -= 1.0
    GROUPS.append((lst, col, lab, y_start, yy + 1.0))
    yy -= 0.9
axC.set_ylim(yy + 0.5, 1.4); axC.set_xlim(0, 4.6)
axC.spines["left"].set_visible(False); axC.set_yticks([]); axC.patch.set_alpha(0)
axC.set_xticks([0, 1, 2, 3, 4]); axC.set_xlabel("$-\\log_{10}$ P", fontsize=6.0, labelpad=1.5)
thr = SMY["ora_calibration"]
for lst, col, lab, y0, y1 in GROUPS:
    t = -np.log10(thr[lst]["p_fwer05_random"])
    axC.plot([t, t], [y1 - 0.45, y0 + 0.45], color="#8C9299", lw=0.7, ls=(0, (2.2, 1.6)), zorder=1)
    axC.text(0.02, y0 + 0.62, lab, ha="left", va="bottom", fontsize=5.8, color=col, fontweight="bold",
             transform=axC.get_yaxis_transform())
for y_, r, col in ROWS:
    x = -np.log10(r.p)
    axC.plot([0, x], [y_, y_], color="#D5D9DE", lw=0.6, zorder=1)
    axC.scatter([x], [y_], s=6 + 7 * r.hits, color=col, zorder=3, linewidth=0)
    gl = [UPPER2NICE.get(g, g) for g in r.genes.split(", ")]
    gtxt = ", ".join(gl) if len(gl) <= 3 else ", ".join(gl[:3]) + f" +{len(gl) - 3}"
    ytxt = 7.5 + 44.5 * (y_ - (yy + 0.5)) / (1.4 - (yy + 0.5))
    C.text(AXX0 - 1.2, ytxt, nice_term(r.term), ha="right", fontsize=5.4)
    C.text(GX, ytxt, gtxt, ha="left", fontsize=5.1, fontstyle="italic", color=col)
C.text(82.2, 57.4, "Top four terms per list; dashed line: family-wise 5% threshold from random gene sets of the same size",
       ha="left", fontsize=5.2, color=MUTED)
C.text(GX, 54.0, "Genes", ha="left", fontsize=5.5, color=INK)
C.save("Figure08_panel_biology")


## 8. Ready-to-paste legend, methods and numbers

In [ ]:
def pf(p): return f"P = {p:.2g}" if p >= 0.001 else f"P = {p:.1e}"
sig_sub = SUBTAB[SUBTAB.q_BH < 0.05].sort_values("p")
best = {k: ORA[ORA.list == k].nsmallest(1, "p").iloc[0] for k in LISTS}
nnb = CT.iloc[0]
TEXT = f"""FIGURE LEGEND - Biology of the Boruta panel
(a) Each panel gene on a dopamine-neuron lineage axis built from Kamath et al. (2022, Supplementary Table 8): the gene's mean
marker z across the six CALB1 subtypes minus its mean across the four SOX6 subtypes (0 when the gene marks no subtype).
Upper tracks, the {len(DOWN)} genes lower and the {len(UP)} genes higher in PD dopamine neurons; below, all {N:,} measured genes (square-root
counts, symmetric-log axis). Shaded, the outer 5% of all genes. Down genes sit on the vulnerable SOX6 side and up genes on the
resilient CALB1 side (Mann-Whitney {pf(SUB['up_vs_down'])}; against gene sets matched on the strength and direction of their PD
change, {pf(SUB['p_vs_DE_matched'])}). (b) Overlap of the down and up genes with the 200 strongest markers of each subtype
(hypergeometric test against the {N:,} measured genes, Benjamini-Hochberg across the 20 tests); filled circles, FDR < 0.05. SOX6_AGTR1
is the population that degenerates in PD. (c) Over-representation of the up, down and all 30 genes among Gene Ontology,
Reactome, KEGG and WikiPathways terms of 10-500 measured genes; the four smallest P values per list, with the genes behind each.
Dashed lines, the family-wise 5% threshold: the P value that random gene sets of the same size reach anywhere among the
{CAL['all']['terms']:,} terms in 5% of draws. No term crosses it, and none reaches FDR < 0.05.

METHODS - Enrichment and pathway analysis
The 30 Boruta genes were split by the sign of their pooled PD effect in the discovery donors ({len(UP)} higher, {len(DOWN)} lower in PD)
and analysed separately, with all 30 as a secondary analysis. The background was the {N:,} genes measured in the discovery data.
Over-representation: gene sets from Enrichr (GO Biological Process, Cellular Component and Molecular Function 2023, Reactome
2022, KEGG 2021, WikiPathways 2023), restricted to measured genes, kept at 10-500 genes, identical sets counted once
({CAL['all']['terms']:,} terms); one-sided hypergeometric tests; Benjamini-Hochberg over every term, not only the terms a list touches. Each
result was calibrated against 2,000 random gene sets of the same size (family-wise threshold) and 2,000 sets matched gene by
gene on the magnitude and direction of the PD effect; with this procedure random sets returned any FDR < 0.05 term in
{100 * max(v['random_sets_with_any_BH_hit'] for v in CAL.values()):.1f}% of draws or fewer.
Dopamine-neuron subtypes: the subtype markers of Kamath et al. (2022; MAST, each of ten subtypes against the other dopamine
neurons, FDR < 0.05) gave each gene a lineage score (mean z over CALB1 subtypes minus mean z over SOX6 subtypes). The scores of
the down and up genes were compared with each other (Mann-Whitney) and against 2,000 gene sets matched on PD effect; the
transcriptome-wide association between each gene's PD effect and its lineage score was tested with 2,000 label shuffles within
study. The 200 strongest markers of each subtype were tested against the down and up lists (hypergeometric, BH over 20 tests).
Pathway neighbours: for every term of 10-150 measured genes holding a panel gene ({len(CT)} terms), the mean PD effect of its other
members, signed by the panel genes' direction, was compared with 5,000 within-study label shuffles; whole-search FWER from the
minimum P per shuffle and empirical FDR. Gene annotation: mygene.info (names, RefSeq summaries) and the Open Targets Platform
(association with Parkinson disease, MONDO_0005180).

RESULTS - numbers
Over-representation: best terms - up: {best['up'].term} (P = {best['up'].p:.1e}, FDR {best['up'].q_BH:.2f}); down: {best['down'].term}
  (P = {best['down'].p:.1e}, FDR {best['down'].q_BH:.2f}); all: {best['all'].term} (P = {best['all'].p:.1e}, FDR {best['all'].q_BH:.2f},
  family-wise P = {best['all'].fwer_random:.3f}). No term at FDR < 0.05 in any list.
Subtypes: down vs up lineage scores {pf(SUB['up_vs_down'])}; down genes vs all genes {pf(SUB['down_vs_background'])}; up genes vs all genes
  {pf(SUB['up_vs_background'])}; beyond DE-matched genes {pf(SUB['p_vs_DE_matched'])}; transcriptome-wide rho = {SUB['rho_transcriptome']:.2f}
  (label-shuffle {pf(SUB['rho_p_label_shuffle'])}){' - the lineage signal is concentrated in the panel, not spread over all genes' if SUB['rho_p_label_shuffle'] > 0.05 else ' - the shift also runs through the whole transcriptome'}.
  Subtype marker overlaps at FDR < 0.05: """ + "; ".join(f"{r.subtype} x {r.list} ({r.hits}: {r.genes}; FDR {r.q_BH:.3f})" for r in sig_sub.itertuples()) + f"""
Pathway neighbours: {len(CT)} terms; best {nnb.term} ({nnb.anchors}; P = {nnb.p:.3f}, FDR {nnb.fdr:.2f}); none at FWER or FDR < 0.05.
Known PD association (Open Targets): """ + "; ".join(f"{r.symbol} {r.pd_association_open_targets:.2f}" + (f" (genetic {r.pd_genetic_association:.2f})" if r.pd_genetic_association > 0 else "")
                                                    for r in GT.sort_values("pd_association_open_targets", ascending=False).head(8).itertuples()) + "\n"
print(TEXT)
open(OUT / "enrich_legend_methods.txt", "w").write(TEXT)
